In [ ]:
!pip install ultralytics 'sng4onnx>=1.0.1' 'onnx_graphsurgeon>=0.3.26' 'ai-edge-litert>=1.2.0' 'onnx>=1.12.0,<2.0.0' 'onnx2tf>=1.26.3,<1.29.0' 'onnxslim>=0.1.71' 'onnxruntime' 'onnxruntime-gpu'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.9/152.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.2/469.2 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.3/237.3 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 MB 5.4 MB/s eta 0:00:00


In [ ]:
!pip install -U ultralytics wandb

input link ke dataset

In [ ]:
!mkdir "/content/dataset"

In [ ]:
!yolo settings wandb=True

✅ Updated 'wandb=True'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/content/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "569f3ba64b326db489132663f79cd37279811de477381b83ac131e6cdd129cbb",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": true,
  "vscode_msg": true,
  "openvino_msg": false
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/quickstart/#ultralytics-settings


In [ ]:
from ultralytics import YOLO
import wandb as wb
import os
from google.colab import userdata


Di tab kanan ada secrets, centang notebook access

In [ ]:
def convert_model(model, fraction=0.06, imgsz=640):
  tflite = model.export(format="tflite", data=f"{dataset_dir}/data.yaml", device="cpu", int8=True, fraction=fraction, imgsz=imgsz)
  return tflite

In [ ]:
class WandbHandler:
  def __init__(self, project_name, root='/content/'):
    self.project_name = project_name
    self.root = root
    wb.login(key=userdata.get('WANDB_API_KEY'))

  def download_from_wandb(self, dataset_name):
    dataset_root = f"{self.root}dataset/{dataset_name}"
    run = wb.init(project=self.project_name, job_type="training")
    artifact = run.use_artifact(f"{self.project_name}/{dataset_name}:latest")

    if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
      print(f"Dataset already exists and is populated at {dataset_root}. Skipping download.")
      return run, dataset_root

    artifact.download(root=dataset_root)
    return run, dataset_root

  def upload_to_wandb(self, dataset_dir):
    artifact = wb.Artifact(name=dataset_dir, type="dataset")
    artifact.add_dir(self.root + dataset_dir)
    with wb.init(project=self.project_name, job_type='upload_dataset') as run:
      run.log_artifact(artifact)

  def update_dataset(self, dataset_dir):
    with wb.init(project=self.project_name, job_type='update_dataset') as run:
      artifact = wb.Artifact(name=dataset_dir, type="dataset")
      artifact.add_dir(self.root + dataset_dir)
      run.log_artifact(artifact)


  def update_file(self, dataset_dir, file_name):
    with wb.init(project=self.project_name, job_type='update_file') as run:
      saved_artifact = run.use_artifact(dataset_dir+":latest")
      draft_artifact = saved_artifact.new_draft()

      draft_artifact.remove(saved_artifact.get_entry(file_name))
      draft_artifact.add_file(local_path= self.root + dataset_dir + "/" + file_name, name=file_name)

      draft_artifact.save()


  def resume_run(self, run_id, dataset_name):
    dataset_root = f"{self.root}dataset/{dataset_name}"
    run = wb.init(project=self.project_name, job_type="training", id=run_id, resume="allow")
    dataset = run.use_artifact(f"{self.project_name}/{dataset_name}:latest")
    checkpoint_root = f"{self.root}{run_id}_checkpoint"
    checkpoint = run.use_artifact(f"{self.project_name}/{run_id}_checkpoint:latest")


    if os.path.exists(checkpoint_root) and len(os.listdir(checkpoint_root)) > 0:
      print(f"Checkpoint already exists and is populated at {checkpoint_root}. Skipping download.")
    else:
      checkpoint.download(root=checkpoint_root)

    if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
      print(f"Dataset already exists and is populated at {dataset_root}. Skipping download.")
    else:
      dataset.download(root=dataset_root)

    return run, checkpoint_root, dataset_root


  def get_model_for_conversion(self, model_name, dataset_name):
    api = wb.Api()
    dataset_root = f"{self.root}dataset/{dataset_name}"
    model_root = f"{self.root}{model_name}"
    dataset = api.artifact(f"{self.project_name}/{dataset_name}:latest")
    model = api.artifact(f"{self.project_name}/{model_name}:latest")

    if os.path.exists(model_root) and len(os.listdir(model_root)) > 0:
      print(f"Model already exists and is populated at {model_root}. Skipping download.")
    else:
      model.download(root=model_root)

    if os.path.exists(dataset_root) and len(os.listdir(dataset_root)) > 0:
      print(f"Dataset already exists and is populated at {dataset_root}. Skipping download.")
    else:
      dataset.download(root=dataset_root)

    return model_root, dataset_root

  def update_mobile_model(self, mobile_dir, model_name):
    with wb.init(project=self.project_name, job_type='update_mobile') as run:
      mobile_name = model_name.split('_')
      mobile_name[-1] = "mobile"
      mobile_name = '_'.join(mobile_name)
      saved_artifact = run.use_artifact(f"{self.project_name}/{mobile_name}:latest")
      draft_artifact = saved_artifact.new_draft()
      draft_artifact.add_file(local_path=mobile_dir)
      run.log_artifact(draft_artifact, aliases=["latest", "best"])



Setting parameter dataset di sini, setting resume True jika ingin melanjutkan run yang sebelumnya sudah dilaksanakan namun terputus, set resume False jika tidak

In [ ]:

project = "FAIt (Food AI-based tracking)"
dataset_name = "Food_Computer_Vision_Dataset" # nama dataset yang suda ada di Weights & Biases
model_name = "yolo26s_Food_Computer_Vision_Dataset_15_2026-03-24-15-44_model"
wandb_handler = WandbHandler(project)
model_dir, dataset_dir = wandb_handler.get_model_for_conversion(model_name, dataset_name)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: b10g004dicoding (b10g004dicoding-dicoding-batch-10-g004) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Model already exists and is populated at /content/yolo26s_Food_Computer_Vision_Dataset_15_2026-03-24-15-44_model. Skipping download.
Dataset already exists and is populated at /content/dataset/Food_Computer_Vision_Dataset. Skipping download.


In [ ]:
model = YOLO(f"{model_dir}/best.pt")
tflite = convert_model(model, fraction=0.06, imgsz=640)

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26s summary (fused): 122 layers, 9,484,530 parameters, 0 gradients, 20.6 GFLOPs

PyTorch: starting from '/content/yolo26s_Food_Computer_Vision_Dataset_15_2026-03-24-15-44_model/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (19.4 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...
TensorFlow SavedModel: collecting INT8 calibration images from 'data=/content/dataset/Food_Computer_Vision_Dataset/data.yaml'
Fast image access ✅ (ping: 0.0±0.0 ms, read: 25.6±12.6 MB/s, size: 27.8 KB)
Scanning /content/dataset/Food_Computer_Vision_Dataset/valid/labels.cache... 285 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 285/285 44.3Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 14, len(boxes) = 347. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segm

/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 18 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 2.9s, saved as '/content/yolo26s_Food_Computer_Vision_Dataset_15_2026-03-24-15-44_model/best.onnx' (36.6 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/content/yolo26s_Food_Computer_Vision_Dataset_15_2026-03-24-15-44_model/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 300, 6), dtype=tf.float32, name=None)
Captures:
  132976080331472: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  132976080329552: TensorSpec(shape=(3, 3, 3, 32), dtype=tf.float32, name=None)
  132976080332432: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  132976080336656: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  132976080337232: TensorSpec(shape=(3, 3, 32, 64), dtype=tf.float32, name=None)
  132976080334352: Tenso

In [ ]:
wandb_handler = WandbHandler(project)
wandb_handler.update_mobile_model(tflite, model_name)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
